# Similarity-aware Label Smoothing

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Dataset Preparation

### MNIST

In [12]:
# mnist_train_ds_raw = datasets.MNIST(root="./data/mnist", train=True, download=True, transform=transforms.ToTensor())

# all_imgs = torch.stack([img for img, _ in mnist_train_ds_raw], dim = 0)
# ds_mean = all_imgs.mean(dim=(0, 2, 3))
# ds_std = all_imgs.std(dim=(0, 2, 3))

# transform = transforms.Compose([
#     transforms.ToTensor(),
#     transforms.Normalize((ds_mean, ), (ds_std, ))
# ])

# mnist_train_ds = datasets.MNIST(root="./data/mnist", train=True, download=True, transform=transform)
# mnist_test_ds = datasets.MNIST(root="./data/mnist", train=False, transform=transform)

# train_loader = DataLoader(mnist_train_ds, batch_size=128, shuffle=True)
# test_loader = DataLoader(mnist_test_ds, batch_size=256)

### CIFAR-100

In [13]:
cifar_train_ds_raw = datasets.CIFAR100(root="./data/cifar", train=True, download=True, transform=transforms.ToTensor())

all_imgs = torch.stack([img for img, _ in cifar_train_ds_raw], dim = 0)
ds_mean = all_imgs.mean(dim=(0, 2, 3))
ds_std = all_imgs.std(dim=(0, 2, 3))

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=ds_mean.tolist(), std=ds_std.tolist())
])

cifar_train_ds = datasets.CIFAR100(root="./data/cifar", train=True, download=True, transform=transform)
cifar_test_ds = datasets.CIFAR100(root="./data/cifar", train=False, transform=transform)

train_loader = DataLoader(cifar_train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(cifar_test_ds, batch_size=256)

## Modeld

### MNIST

In [14]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2,2)
        self.fc1   = nn.Linear(64*7*7, 128)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


### CIFAR-100

In [15]:
def RESNET18(pretrained=False):
    model = models.resnet18(pretrained=pretrained)

    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 100)

    return model

## Training Utils

### Accuracy Measure

In [22]:
def accuracy(model, loader, k = (1, 5)):
    model.eval()
    correct = {key:0 for key in k}
    total = 0

    maxk = max(k)

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)

            _, pred = outputs.topk(maxk, dim=1, largest=True, sorted=True)

            y_expand = y.view(-1, 1).expand_as(pred)

            for key in k:
                correct[key] += (pred[:, :key] == y.view(-1, 1)).any(dim=1).sum().item()
            total += y.size(0)

    return {key: correct[key] / total for key in k}

### Label Smoothing

In [17]:
def uniform_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return ((num_classes * (1 - epsilon) - 1) * y_onehot + epsilon) / (num_classes - 1)

## Training Loop

In [24]:
def train(model, train_loader, test_loader, optimizer=None, scheduler=None, num_classes=100, epochs=10, epsilon=0.1):
    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for x, y in tqdm(train_loader, leave=False):
            x, y = x.to(device), y.to(device)

            targets = uniform_smooth_labels(y, num_classes=num_classes, epsilon=epsilon)

            optimizer.zero_grad()
            logits = model(x)
            loss = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)

        if scheduler is not None: scheduler.step()

        test_acc = accuracy(model, test_loader)
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {running_loss/len(train_loader.dataset):.4f} | Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")

## RUN Experiments

In [25]:
model = RESNET18().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    num_classes=100,
    epochs=50,
    epsilon=0,
)


Epoch 1/10 | Loss: 3.9555 | Test Acc: 0.1405 | Top-5 Test Acc: 0.3954


Epoch 2/10 | Loss: 3.2186 | Test Acc: 0.2515 | Top-5 Test Acc: 0.5416


Epoch 3/10 | Loss: 2.6727 | Test Acc: 0.3351 | Top-5 Test Acc: 0.6587


Epoch 4/10 | Loss: 2.1965 | Test Acc: 0.3852 | Top-5 Test Acc: 0.7018


Epoch 5/10 | Loss: 1.8242 | Test Acc: 0.4346 | Top-5 Test Acc: 0.7533


Epoch 6/10 | Loss: 1.4729 | Test Acc: 0.4839 | Top-5 Test Acc: 0.7898


Epoch 7/10 | Loss: 1.0793 | Test Acc: 0.5035 | Top-5 Test Acc: 0.7978


Epoch 8/10 | Loss: 0.5900 | Test Acc: 0.5429 | Top-5 Test Acc: 0.8304


Epoch 9/10 | Loss: 0.1928 | Test Acc: 0.5875 | Top-5 Test Acc: 0.8512


Epoch 10/10 | Loss: 0.0557 | Test Acc: 0.5955 | Top-5 Test Acc: 0.8523
